<a href="https://colab.research.google.com/github/wnstj1126-debug/-/blob/main/M0_BRIDGE_2048_APPLY_v1_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# M0_BRIDGE_2048_APPLY_v1.1

**목적**: v1에서 발생한 `APPLY_SCOPE_MANIFEST_BUG`를 수정하고,
FEMTO 전체 28개 record에서 각 10개씩 총 280개를 Bridge 잠금 사양으로 변환·검증한다.

## v1 오류 원인

```
source_classification_manifest.csv  ← Learning 전용
→ 파일이 존재해서 첫 분기 실행
→ TEST/FULL_TEST 누락
→ 60개만 처리
```

## v1.1 수정

```
femto_csv_source_manifest.csv  ← 43,369개 전체
→ 이것을 모집단으로 사용
→ 28 records × 10 = 280개
```

## 예상 표본

| Split | 표본 |
|:---|---:|
| LEARNING | 60 |
| TEST | 110 |
| FULL_TEST | 110 |
| **합계** | **280** |

## 절대 원칙

- 기존 `bridge_spec_locked.json`, `APPLY_FINAL_SUMMARY.json` 수정 **금지**
- TEST/FULL_TEST로 Bridge 방식·기준 변경 **금지**
- `APPLY_SCOPE_MANIFEST_BUG`가 원인이므로 기존 잠금 기준 그대로 적용
- 원본 CSV 수정·삭제·이동 **금지**

## 실행 순서

1. **셀 01** — 단위 검증
2. **셀 02** — 설정 + 공통 함수
3. **셀 03** — Gate 확인
4. **셀 04** — 전체 manifest + 범위 검증
5. **셀 05** — 280개 표본 선택
6. **셀 06** — 변환 + 품질 평가 + 대역 진단
7. **셀 07** — 결정론 재현성
8. **셀 08** — 원본 불변성
9. **셀 09** — 최종 판정
10. **셀 10** — 결과 확인

In [1]:
# ================================================================
# 셀 01 — 단위 검증
# ================================================================

import re
import numpy as np
import scipy.signal

# ── 숫자 검증 ─────────────────────────────────────────────────
assert 8384 + 15462 + 19523 == 43369,  "전체 CSV 합계 오류"
assert 850  + 1335  + 2168  == 4353,   "temp 합계 오류"
assert 7534 + 14127 + 17355 == 39016,  "진동 합계 오류"
assert 6 + 11 + 11 == 28,             "record 합계 오류"
assert 60 + 110 + 110 == 280,          "표본 합계 오류"
print("[Unit] 숫자 assert 통과 ✅")

# ── resample_poly ─────────────────────────────────────────────
_x = np.ones(2560, dtype=np.float64)
_y = scipy.signal.resample_poly(_x, up=4, down=5,
                                  window=("kaiser", 5.0), padtype="line")
assert len(_y) == 2048
print("[Unit] resample_poly(2560→2048) ✅")

# ── 결정론 ────────────────────────────────────────────────────
_sig = np.random.default_rng(42).standard_normal(2560)
_r1  = scipy.signal.resample_poly(_sig, 4, 5, window=("kaiser",5.0), padtype="line")
_r2  = scipy.signal.resample_poly(_sig, 4, 5, window=("kaiser",5.0), padtype="line")
assert np.array_equal(_r1, _r2)
print("[Unit] 결정론 ✅")

# ── natural sort ─────────────────────────────────────────────
def _nk(s):
    return [int(t) if t.isdigit() else t.lower()
            for t in re.split(r"(\d+)", str(s))]
assert sorted(["acc_10.csv","acc_2.csv","acc_1.csv"], key=_nk) == \
       ["acc_1.csv","acc_2.csv","acc_10.csv"]
print("[Unit] natural sort ✅")

# ── record_key 중복 허용 (TEST vs FULL_TEST 동명 bearing) ────
_keys = ["TEST/Bearing1_3", "FULL_TEST/Bearing1_3"]
assert len(set(_keys)) == 2
print("[Unit] record_key 중복 구분 ✅")

# ── normalize_split 검증 ──────────────────────────────────────
_aliases = {
    "LEARNING": "LEARNING", "Training(Learning)_set": "LEARNING",
    "TEST": "TEST", "Test(Test)_set": "TEST",
    "FULL_TEST": "FULL_TEST", "Validation(Full_Test)_Set": "FULL_TEST",
}
def _norm(s):
    v = str(s).strip().upper().replace("-","_").replace(" ","_")
    tbl = {
        "LEARNING":"LEARNING","TRAINING":"LEARNING",
        "TRAINING(LEARNING)_SET":"LEARNING",
        "TEST":"TEST","TEST(TEST)_SET":"TEST",
        "FULL_TEST":"FULL_TEST","FULLTEST":"FULL_TEST",
        "VALIDATION":"FULL_TEST","VALIDATION(FULL_TEST)_SET":"FULL_TEST",
    }
    if v in tbl: return tbl[v]
    if "FULL" in v or "VALIDATION" in v: return "FULL_TEST"
    if "LEARNING" in v or "TRAINING" in v: return "LEARNING"
    if "TEST" in v: return "TEST"
    return None
for raw, expected in _aliases.items():
    got = _norm(raw)
    assert got == expected, f"{raw!r} → {got} (기대 {expected})"
print("[Unit] normalize_split ✅")

# ── np.trapezoid fallback ────────────────────────────────────
try:
    _tv = np.trapezoid([1.,2.,3.], [0.,.5,1.])
    print("[Unit] np.trapezoid ✅")
except AttributeError:
    _tv = np.trapz([1.,2.,3.], [0.,.5,1.])
    print("[Unit] np.trapz (fallback) ✅")

print("\n[Unit] 모든 단위 검증 통과 ✅")

[Unit] 숫자 assert 통과 ✅
[Unit] resample_poly(2560→2048) ✅
[Unit] 결정론 ✅
[Unit] natural sort ✅
[Unit] record_key 중복 구분 ✅
[Unit] normalize_split ✅
[Unit] np.trapezoid ✅

[Unit] 모든 단위 검증 통과 ✅


In [2]:
# ================================================================
# 셀 02 — 설정 + 공통 함수
# ================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import hashlib, json, os, re, sys, traceback
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import scipy, scipy.signal, scipy.stats
from scipy.signal import periodogram
from scipy.stats  import kurtosis as _kurt, skew as _skew

VERSION = "M0_BRIDGE_2048_APPLY_v1_1"

# ── 고정 상수 ─────────────────────────────────────────────────
SOURCE_FS, SOURCE_SAMPLES = 25600, 2560
OUTPUT_FS, OUTPUT_SAMPLES = 20480, 2048
RESAMPLE_UP, RESAMPLE_DOWN = 4, 5
KAISER_BETA = 5.0
PADTYPE     = "line"
COL_H       = 4
COMMON_BAND = (0, 6000)

EXPECTED_ALL        = {"LEARNING":8384,  "TEST":15462, "FULL_TEST":19523}
EXPECTED_TEMP       = {"LEARNING":850,   "TEST":1335,  "FULL_TEST":2168}
EXPECTED_VIBRATION  = {"LEARNING":7534,  "TEST":14127, "FULL_TEST":17355}
EXPECTED_RECORDS    = {"LEARNING":6,     "TEST":11,    "FULL_TEST":11}
EXPECTED_SAMPLE_CNT = {"LEARNING":60,    "TEST":110,   "FULL_TEST":110}
EXPECTED_TOTAL_CSV  = 43369
EXPECTED_TOTAL_TEMP = 4353
EXPECTED_TOTAL_VIB  = 39016
EXPECTED_TOTAL_REC  = 28
EXPECTED_TOTAL_SMP  = 280

QUALITY_CRITERIA = {
    "energy_0_6khz_relerr_median": 0.05,
    "energy_0_6khz_relerr_p95":    0.10,
    "dominant_freq_abserr_median": 50.0,
    "rms_relerr_median":           0.05,
}
RMS_WARN_THR = 0.10
REPRO_PER_SPLIT = 4   # 결정론 표본 (LEARNING/TEST/FULL_TEST 각 4개)

TEMP_RE = re.compile(r"^temp_\d+\.csv$", re.IGNORECASE)

PROJECT_ROOT = next(
    (c for c in [
        Path("/content/drive/MyDrive/Colab Notebooks/field_iis3dwb"),
        Path("/content/drive/My Drive/Colab Notebooks/field_iis3dwb"),
    ] if c.exists()), None
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("PROJECT_ROOT_NOT_FOUND")

RUN_ID     = datetime.now().strftime("%Y%m%d_%H%M%S_bridge_apply_v1_1")
OUTPUT_DIR = PROJECT_ROOT / "bridge_outputs" / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
ARRAYS_DIR = OUTPUT_DIR / "confirmation_arrays"
ARRAYS_DIR.mkdir()

print(f"VERSION      : {VERSION}")
print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"OUTPUT_DIR   : {OUTPUT_DIR}")


# ── 공통 함수 ─────────────────────────────────────────────────
def write_json(path, obj):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=str)
    os.replace(tmp, path)

def natural_key(s):
    return [int(t) if t.isdigit() else t.lower()
            for t in re.split(r"(\d+)", str(s))]

def sha256_file(p, chunk=1<<20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        while True:
            c = f.read(chunk)
            if not c: break
            h.update(c)
    return h.hexdigest()

def sha256_bytes(arr):
    return hashlib.sha256(arr.astype(np.float64).tobytes()).hexdigest()

def sha256_bytes32(arr):
    return hashlib.sha256(arr.astype(np.float32).tobytes()).hexdigest()

def rel_err(a, b):
    if b is None or abs(b) < 1e-12: return None
    return abs(a - b) / abs(b)

try:
    _trap = np.trapezoid
except AttributeError:
    _trap = np.trapz

def band_energy(freqs, psd, flo, fhi):
    mask = (freqs >= flo) & (freqs <= fhi)
    return float(_trap(psd[mask], freqs[mask])) if mask.any() else 0.0

def dominant_freq(freqs, psd, flo, fhi):
    mask = (freqs >= flo) & (freqs <= fhi)
    return float(freqs[mask][np.argmax(psd[mask])]) if mask.any() else None

def compute_metrics(sig, fs):
    rms   = float(np.sqrt(np.mean(sig**2)))
    peak  = float(np.max(np.abs(sig)))
    crest = peak / rms if rms > 1e-12 else None
    freqs, psd = periodogram(sig, fs=fs, window="hann",
                             detrend="constant", scaling="density")
    e06  = band_energy(freqs, psd, *COMMON_BAND)
    domf = dominant_freq(freqs, psd, *COMMON_BAND)
    # 진단 대역
    bands = {
        "e_0_2":   band_energy(freqs, psd, 0,     2000),
        "e_2_4":   band_energy(freqs, psd, 2000,  4000),
        "e_4_6":   band_energy(freqs, psd, 4000,  6000),
        "e_6_8":   band_energy(freqs, psd, 6000,  8000),
        "e_8_10":  band_energy(freqs, psd, 8000,  int(fs*0.4)),
        "e_10_nyq":band_energy(freqs, psd, int(fs*0.4), int(fs*0.5)),
    }
    return {
        "rms":rms, "peak":peak, "crest":crest,
        "kurtosis":float(_kurt(sig)), "skew":float(_skew(sig)),
        "energy_0_6khz":e06, "dominant_freq":domf, **bands
    }

def convert_one(sig_src):
    assert len(sig_src) == SOURCE_SAMPLES
    assert np.isfinite(sig_src).all()
    out = scipy.signal.resample_poly(
        sig_src.astype(np.float64),
        up=RESAMPLE_UP, down=RESAMPLE_DOWN,
        window=("kaiser", KAISER_BETA), padtype=PADTYPE,
    )
    assert len(out) == OUTPUT_SAMPLES
    return out

def load_h(path):
    df = pd.read_csv(path, header=None, low_memory=False)
    return pd.to_numeric(df.iloc[:, COL_H], errors="coerce").to_numpy(dtype=np.float64)

def safe_median(s): return float(s.dropna().median()) if len(s.dropna())>0 else None
def safe_p95(s):    return float(np.percentile(s.dropna(),95)) if len(s.dropna())>0 else None

def find_latest(pattern):
    dirs = sorted(
        [p for p in (PROJECT_ROOT/"bridge_outputs").glob(pattern) if p.is_dir()],
        key=lambda p: p.stat().st_mtime_ns, reverse=True,
    )
    return dirs[0] if dirs else None

def normalize_split(s):
    v = str(s).strip().upper().replace("-","_").replace(" ","_")
    tbl = {
        "LEARNING":"LEARNING","TRAINING":"LEARNING",
        "TRAINING(LEARNING)_SET":"LEARNING",
        "TEST":"TEST","TEST(TEST)_SET":"TEST",
        "FULL_TEST":"FULL_TEST","FULLTEST":"FULL_TEST",
        "VALIDATION":"FULL_TEST","VALIDATION(FULL_TEST)_SET":"FULL_TEST",
    }
    if v in tbl: return tbl[v]
    if "FULL" in v or "VALIDATION" in v: return "FULL_TEST"
    if "LEARNING" in v or "TRAINING" in v: return "LEARNING"
    if "TEST" in v: return "TEST"
    return None

def require_col(df, candidates, name):
    for c in candidates:
        if c in df.columns: return c
    raise KeyError(f"REQUIRED_COLUMN_NOT_FOUND: {name}; have={list(df.columns)}")

def select_unique_linspace(total, n=10):
    if total < n:
        raise ValueError(f"NOT_ENOUGH_FILES: total={total}, required={n}")
    targets = np.linspace(0, total-1, n)
    seen, result = set(), []
    for t in targets:
        first = int(round(float(t)))
        if first not in seen:
            seen.add(first); result.append(first); continue
        cands = sorted(
            (i for i in range(total) if i not in seen),
            key=lambda i: (abs(i-t), i),
        )
        if not cands: raise RuntimeError("UNIQUE_LINSPACE_SELECTION_FAILED")
        seen.add(cands[0]); result.append(cands[0])
    assert len(result)==n and len(set(result))==n
    return result

def write_fatal(stage, exc):
    write_json(OUTPUT_DIR / "fatal_error.json", {
        "stage": stage, "exception_type": type(exc).__name__,
        "exception_message": str(exc),
        "traceback": traceback.format_exc(),
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "output_directory": str(OUTPUT_DIR),
    })

print(f"NumPy {np.__version__}  SciPy {scipy.__version__}  Python {sys.version.split()[0]}")
print("[SETUP] 완료")

Mounted at /content/drive
VERSION      : M0_BRIDGE_2048_APPLY_v1_1
PROJECT_ROOT : /content/drive/MyDrive/Colab Notebooks/field_iis3dwb
OUTPUT_DIR   : /content/drive/MyDrive/Colab Notebooks/field_iis3dwb/bridge_outputs/20260725_194318_bridge_apply_v1_1
NumPy 2.0.2  SciPy 1.16.3  Python 3.12.13
[SETUP] 완료


In [3]:
# ================================================================
# 셀 03 — Gate 확인
# ================================================================

APPLY_V1_DIR = find_latest("*_bridge_apply_v1")
if APPLY_V1_DIR is None:
    raise FileNotFoundError("BRIDGE_APPLY_V1_NOT_FOUND")

PREP_DIR = find_latest("*_bridge_prep_v1_1")
if PREP_DIR is None:
    raise FileNotFoundError("BRIDGE_PREP_V1_1_NOT_FOUND")

print(f"APPLY_V1_DIR : {APPLY_V1_DIR}")
print(f"PREP_DIR     : {PREP_DIR}")

v1_summary  = json.loads((APPLY_V1_DIR / "APPLY_FINAL_SUMMARY.json").read_text("utf-8"))
v1_gate     = json.loads((APPLY_V1_DIR / "apply_gate_check.json").read_text("utf-8"))
contract    = json.loads((APPLY_V1_DIR / "bridge_implementation_contract.json").read_text("utf-8"))
spec_locked = json.loads((PREP_DIR / "bridge_spec_locked.json").read_text("utf-8"))
prep_summ   = json.loads((PREP_DIR / "FINAL_BRIDGE_PREP_SUMMARY.json").read_text("utf-8"))

# parent_bridge_spec_sha256 일치 확인
spec_locked_sha_now = sha256_file(PREP_DIR / "bridge_spec_locked.json")
contract_sha = contract.get("parent_bridge_spec_sha256", "")

gate_conditions = {
    "v1_gate_pass":              bool(v1_gate.get("gate_pass") is True),
    "v1_quality_pass":           bool(v1_summary.get("quality_pass") is True),
    "v1_determinism_pass":       bool(v1_summary.get("determinism_pass") is True),
    "v1_source_unchanged":       bool(v1_summary.get("source_files_unchanged") is True),
    "bridge_spec_ready":         spec_locked.get("status") == "BRIDGE_SPEC_READY",
    "selected_bridge":           spec_locked.get("selected_bridge") == "DURATION_PRESERVING_2048",
    "parent_sha256_match":       contract_sha == spec_locked_sha_now,
    "source_fs_25600":           int(spec_locked.get("source_fs_hz",0)) == 25600,
    "output_fs_20480":           int(spec_locked.get("output_fs_hz",0)) == 20480,
    "source_length_2560":        int(spec_locked.get("source_sample_count",0)) == 2560,
    "output_length_2048":        int(spec_locked.get("output_sample_count",0)) == 2048,
    "up_4":                      int(spec_locked.get("resample_up",0)) == 4,
    "down_5":                    int(spec_locked.get("resample_down",0)) == 5,
    "window_kaiser":             contract.get("window_type") == "kaiser",
    "beta_5":                    float(contract.get("window_beta",0)) == 5.0,
    "padtype_line":              contract.get("padtype") == "line",
    "column_index_4":            int(spec_locked.get("column_index",0)) == 4,
}

GATE_PASS = all(gate_conditions.values())

write_json(OUTPUT_DIR / "apply_v1_1_gate_check.json", {
    "version": VERSION, "created_at": datetime.now().isoformat(timespec="seconds"),
    "gate_pass": GATE_PASS,
    "gate_conditions": gate_conditions,
    "apply_v1_dir": str(APPLY_V1_DIR),
    "prep_dir": str(PREP_DIR),
})

print("=== Bridge Gate ===")
for k, v in gate_conditions.items():
    print(f"  {'✅' if v else '❌'} {k}: {v}")
print(f"  GATE_PASS = {GATE_PASS}")

if not GATE_PASS:
    write_fatal("03_GATE", RuntimeError("APPLY_V1_1_BLOCKED_BY_GATE"))
    raise RuntimeError("APPLY_V1_1_BLOCKED_BY_GATE")

APPLY_V1_DIR : /content/drive/MyDrive/Colab Notebooks/field_iis3dwb/bridge_outputs/20260725_185448_bridge_apply_v1
PREP_DIR     : /content/drive/MyDrive/Colab Notebooks/field_iis3dwb/bridge_outputs/20260725_192744_bridge_prep_v1_1
=== Bridge Gate ===
  ✅ v1_gate_pass: True
  ✅ v1_quality_pass: True
  ✅ v1_determinism_pass: True
  ✅ v1_source_unchanged: True
  ✅ bridge_spec_ready: True
  ✅ selected_bridge: True
  ✅ parent_sha256_match: True
  ✅ source_fs_25600: True
  ✅ output_fs_20480: True
  ✅ source_length_2560: True
  ✅ output_length_2048: True
  ✅ up_4: True
  ✅ down_5: True
  ✅ window_kaiser: True
  ✅ beta_5: True
  ✅ padtype_line: True
  ✅ column_index_4: True
  GATE_PASS = True


In [4]:
# ================================================================
# 셀 04 — 전체 manifest 로드 + 범위 검증
#
# ★ APPLY_SCOPE_MANIFEST_BUG 수정 핵심 셀
#   source_classification_manifest.csv (Learning 전용) 대신
#   femto_csv_source_manifest.csv (43,369개 전체) 를 모집단으로 사용
# ================================================================

# ── 전체 manifest 탐색 ────────────────────────────────────────
def find_full_manifest(project_root):
    bridge_root = project_root / "bridge_outputs"
    cands = sorted(
        [p for p in bridge_root.glob("*_source_lock_v1_1/femto_csv_source_manifest.csv")
         if p.is_file()],
        key=lambda p: p.stat().st_mtime_ns, reverse=True,
    )
    if not cands:
        cands = sorted(
            [p for p in bridge_root.rglob("femto_csv_source_manifest.csv")
             if p.is_file() and "_checkpoints" not in p.parts],
            key=lambda p: p.stat().st_mtime_ns, reverse=True,
        )
    if not cands:
        raise FileNotFoundError("FULL_FEMTO_SOURCE_MANIFEST_NOT_FOUND")
    return cands[0]

SOURCE_MANIFEST_PATH = find_full_manifest(PROJECT_ROOT)
print(f"[SCOPE] manifest: {SOURCE_MANIFEST_PATH}")

full_df = pd.read_csv(SOURCE_MANIFEST_PATH, low_memory=False)
print(f"[SCOPE] raw rows: {len(full_df):,}")

# 필수 열 탐지
split_col    = require_col(full_df, ["split","dataset_split","split_name"],           "split")
bearing_col  = require_col(full_df, ["logical_bearing_id","bearing_id","bearing"],     "logical_bearing_id")
filename_col = require_col(full_df, ["csv_file_name","file_name","filename"],           "csv_file_name")
path_col     = require_col(full_df, ["absolute_path","source_path","file_path"],        "absolute_path")
sequence_col = next((c for c in ["sequence_index","source_sequence_index","index_in_record"]
                     if c in full_df.columns), None)

# 정규화
full_df["split_normalized"]      = full_df[split_col].map(normalize_split)
full_df["logical_bearing_id_n"]  = full_df[bearing_col].astype(str).str.strip()
full_df["csv_file_name_n"]        = full_df[filename_col].astype(str).str.strip()
full_df["absolute_path_n"]        = full_df[path_col].astype(str).str.strip()

if full_df["split_normalized"].isna().any():
    bad = full_df.loc[full_df["split_normalized"].isna(), split_col].value_counts().to_dict()
    raise RuntimeError(f"UNKNOWN_SPLIT_VALUES: {bad}")

full_df["is_temp"]      = full_df["csv_file_name_n"].map(lambda x: bool(TEMP_RE.fullmatch(x)))
full_df["file_exists"]  = full_df["absolute_path_n"].map(lambda x: Path(x).is_file())

if "file_size_bytes" in full_df.columns:
    full_df["nonzero_file"] = pd.to_numeric(full_df["file_size_bytes"], errors="coerce").fillna(0) > 0
else:
    full_df["nonzero_file"] = full_df["absolute_path_n"].map(
        lambda x: Path(x).stat().st_size > 0 if Path(x).is_file() else False
    )

if "is_valid_csv_candidate" in full_df.columns:
    valid_cand = full_df["is_valid_csv_candidate"].astype(str).str.strip().str.lower().isin(["true","1","yes"])
else:
    valid_cand = pd.Series(True, index=full_df.index)

full_df["suffix_n"] = full_df["csv_file_name_n"].map(lambda x: Path(x).suffix.lower())
full_df["is_vibration"] = (
    (full_df["suffix_n"] == ".csv")
    & (~full_df["is_temp"])
    & full_df["file_exists"]
    & full_df["nonzero_file"]
    & valid_cand
)

full_df["record_key"] = full_df["split_normalized"] + "/" + full_df["logical_bearing_id_n"]

vibration_df = full_df[full_df["is_vibration"]].copy()

# ── 범위 숫자 검증 ────────────────────────────────────────────
all_cnt  = full_df.groupby("split_normalized").size().to_dict()
temp_cnt = full_df[full_df["is_temp"]].groupby("split_normalized").size().to_dict()
vib_cnt  = vibration_df.groupby("split_normalized").size().to_dict()
rec_cnt  = vibration_df.groupby("split_normalized")["record_key"].nunique().to_dict()

scope_checks = {}
for sp in ["LEARNING","TEST","FULL_TEST"]:
    scope_checks[f"{sp}_all"]       = int(all_cnt.get(sp,0))  == EXPECTED_ALL[sp]
    scope_checks[f"{sp}_temp"]      = int(temp_cnt.get(sp,0)) == EXPECTED_TEMP[sp]
    scope_checks[f"{sp}_vibration"] = int(vib_cnt.get(sp,0))  == EXPECTED_VIBRATION[sp]
    scope_checks[f"{sp}_records"]   = int(rec_cnt.get(sp,0))  == EXPECTED_RECORDS[sp]
scope_checks["total_csv"]       = len(full_df)        == EXPECTED_TOTAL_CSV
scope_checks["total_temp"]      = int(full_df["is_temp"].sum()) == EXPECTED_TOTAL_TEMP
scope_checks["total_vibration"] = len(vibration_df)   == EXPECTED_TOTAL_VIB
scope_checks["total_records"]   = vibration_df["record_key"].nunique() == EXPECTED_TOTAL_REC
scope_checks["no_temp_in_vib"]  = not vibration_df["is_temp"].any()
scope_checks["all_vib_exist"]   = bool(vibration_df["file_exists"].all())

SOURCE_SCOPE_PASS = all(scope_checks.values())

scope_report = {
    "version": VERSION, "source_manifest_path": str(SOURCE_MANIFEST_PATH),
    "source_manifest_row_count": int(len(full_df)),
    "all_counts":       {k:int(v) for k,v in all_cnt.items()},
    "temperature_counts":{k:int(v) for k,v in temp_cnt.items()},
    "vibration_counts": {k:int(v) for k,v in vib_cnt.items()},
    "record_counts":    {k:int(v) for k,v in rec_cnt.items()},
    "scope_checks": scope_checks,
    "source_scope_pass": SOURCE_SCOPE_PASS,
}
write_json(OUTPUT_DIR / "full_source_scope_check.json", scope_report)

print("=== Scope 검증 ===")
for k, v in scope_checks.items():
    print(f"  {'✅' if v else '❌'} {k}")
print(f"  SOURCE_SCOPE_PASS = {SOURCE_SCOPE_PASS}")

if not SOURCE_SCOPE_PASS:
    write_fatal("04_SCOPE", RuntimeError("APPLY_V1_1_SOURCE_SCOPE_MISMATCH"))
    raise RuntimeError("APPLY_V1_1_SOURCE_SCOPE_MISMATCH")

vibration_df.to_csv(OUTPUT_DIR / "vibration_source_manifest_39016.csv",
                    index=False, encoding="utf-8-sig")
print(f"\n진동 CSV: {len(vibration_df):,}개 → vibration_source_manifest_39016.csv 저장")

[SCOPE] manifest: /content/drive/MyDrive/Colab Notebooks/field_iis3dwb/bridge_outputs/20260725_150815_source_lock_v1_1/femto_csv_source_manifest.csv
[SCOPE] raw rows: 43,369
=== Scope 검증 ===
  ✅ LEARNING_all
  ✅ LEARNING_temp
  ✅ LEARNING_vibration
  ✅ LEARNING_records
  ✅ TEST_all
  ✅ TEST_temp
  ✅ TEST_vibration
  ✅ TEST_records
  ✅ FULL_TEST_all
  ✅ FULL_TEST_temp
  ✅ FULL_TEST_vibration
  ✅ FULL_TEST_records
  ✅ total_csv
  ✅ total_temp
  ✅ total_vibration
  ✅ total_records
  ✅ no_temp_in_vib
  ✅ all_vib_exist
  SOURCE_SCOPE_PASS = True

진동 CSV: 39,016개 → vibration_source_manifest_39016.csv 저장


In [5]:
# ================================================================
# 셀 05 — 280개 표본 선택
# ================================================================

selected_rows = []
record_keys   = sorted(vibration_df["record_key"].unique(), key=natural_key)

for rk in record_keys:
    sp, bearing_id = rk.split("/", 1)
    rdf = vibration_df[vibration_df["record_key"] == rk].copy()

    # natural sort
    if sequence_col and sequence_col in rdf.columns:
        rdf["_seq"] = pd.to_numeric(rdf[sequence_col], errors="coerce")
        if rdf["_seq"].notna().all():
            rdf = rdf.sort_values(["_seq","csv_file_name_n"], kind="stable")
        else:
            rdf = rdf.iloc[sorted(range(len(rdf)), key=lambda i: natural_key(rdf.iloc[i]["csv_file_name_n"]))]
    else:
        rdf = rdf.iloc[sorted(range(len(rdf)), key=lambda i: natural_key(rdf.iloc[i]["csv_file_name_n"]))]
    rdf = rdf.reset_index(drop=True)

    indices = select_unique_linspace(len(rdf), 10)
    for pos, idx in enumerate(indices):
        row = rdf.iloc[idx]
        selected_rows.append({
            "sample_id":            f"{sp}_{bearing_id}_{pos:02d}",
            "split":                sp,
            "record_key":           rk,
            "logical_bearing_id":   bearing_id,
            "selection_position":   pos,
            "selected_source_index": int(idx),
            "record_vibration_count": int(len(rdf)),
            "csv_file_name":        row["csv_file_name_n"],
            "absolute_path":        row["absolute_path_n"],
            "source_sequence_index": row[sequence_col] if sequence_col else idx,
            "is_temperature":       False,
            "selection_purpose":    "POST_LOCK_APPLICABILITY_CONFIRMATION",
        })

SAMPLE_DF = pd.DataFrame(selected_rows)

# SHA-256 사전 기록
sha_map = {}
print("[SAMPLE] SHA-256 계산 중...")
for row in SAMPLE_DF.itertuples():
    try:
        sha_map[row.absolute_path] = sha256_file(row.absolute_path)
    except Exception as e:
        sha_map[row.absolute_path] = f"ERROR:{e}"
SAMPLE_DF["source_sha256"] = SAMPLE_DF["absolute_path"].map(sha_map)

# 선택 검증
cnt_by_split  = SAMPLE_DF.groupby("split").size().to_dict()
cnt_per_rec   = SAMPLE_DF.groupby("record_key").size()

sel_checks = {
    "total_280":          len(SAMPLE_DF) == EXPECTED_TOTAL_SMP,
    "unique_paths_280":   SAMPLE_DF["absolute_path"].nunique() == EXPECTED_TOTAL_SMP,
    "record_count_28":    SAMPLE_DF["record_key"].nunique() == EXPECTED_TOTAL_REC,
    "ten_per_record":     bool((cnt_per_rec == 10).all()),
    "learning_60":        int(cnt_by_split.get("LEARNING",0)) == 60,
    "test_110":           int(cnt_by_split.get("TEST",0))     == 110,
    "full_test_110":      int(cnt_by_split.get("FULL_TEST",0))== 110,
    "no_temperature":     not SAMPLE_DF["is_temperature"].any(),
    "all_files_exist":    bool(SAMPLE_DF["absolute_path"].map(lambda x: Path(x).is_file()).all()),
}
SELECTION_PASS = all(sel_checks.values())

print("=== 표본 선택 검증 ===")
for k, v in sel_checks.items():
    print(f"  {'✅' if v else '❌'} {k}")

if not SELECTION_PASS:
    write_fatal("05_SELECTION", RuntimeError("APPLY_V1_1_SELECTION_FAILURE"))
    raise RuntimeError("APPLY_V1_1_SELECTION_FAILURE")

SAMPLE_DF.to_csv(OUTPUT_DIR / "apply_v1_1_sample_manifest.csv",
                 index=False, encoding="utf-8-sig")
print(f"\nL={cnt_by_split.get('LEARNING')}  T={cnt_by_split.get('TEST')}  FT={cnt_by_split.get('FULL_TEST')}  total={len(SAMPLE_DF)}")
print("[SAMPLE] apply_v1_1_sample_manifest.csv 저장 완료")

[SAMPLE] SHA-256 계산 중...
=== 표본 선택 검증 ===
  ✅ total_280
  ✅ unique_paths_280
  ✅ record_count_28
  ✅ ten_per_record
  ✅ learning_60
  ✅ test_110
  ✅ full_test_110
  ✅ no_temperature
  ✅ all_files_exist

L=60  T=110  FT=110  total=280
[SAMPLE] apply_v1_1_sample_manifest.csv 저장 완료


In [6]:
# ================================================================
# 셀 06 — 변환 + 품질 평가 + 대역 진단
# ================================================================

metric_rows, fail_rows, npy_manifest = [], [], []
TOTAL_CNT = len(SAMPLE_DF)

for i, srow in SAMPLE_DF.iterrows():
    fpath = Path(srow["absolute_path"])
    sid   = srow["sample_id"]
    base  = {
        "sample_id":          sid,
        "split":              srow["split"],
        "record_key":         srow["record_key"],
        "logical_bearing_id": srow["logical_bearing_id"],
        "csv_file_name":      srow["csv_file_name"],
        "sequence_index":     srow["source_sequence_index"],
    }
    try:
        src    = load_h(fpath)
        src_len = len(src)
        src_nan = int(np.isnan(src).sum())
        src_inf = int(np.isinf(src).sum())
        if src_len != SOURCE_SAMPLES or src_nan > 0 or src_inf > 0:
            raise ValueError(f"source schema: len={src_len} nan={src_nan} inf={src_inf}")

        out     = convert_one(src)
        out_nan = int(np.isnan(out).sum())
        out_inf = int(np.isinf(out).sum())
        if out_nan > 0 or out_inf > 0:
            raise ValueError(f"output NaN/Inf: nan={out_nan} inf={out_inf}")

        ms = compute_metrics(src, SOURCE_FS)
        mo = compute_metrics(out, OUTPUT_FS)

        rms_re = rel_err(mo["rms"],          ms["rms"])
        e_re   = rel_err(mo["energy_0_6khz"],ms["energy_0_6khz"])
        df_ae  = abs(mo["dominant_freq"]-ms["dominant_freq"]) \
                 if None not in [mo["dominant_freq"],ms["dominant_freq"]] else None
        peak_re  = rel_err(mo["peak"],  ms["peak"])
        crest_re = rel_err(mo["crest"], ms["crest"]) \
                   if None not in [mo["crest"],ms["crest"]] else None

        row = {
            **base,
            "source_length": SOURCE_SAMPLES, "output_length": OUTPUT_SAMPLES,
            "source_fs_hz":  SOURCE_FS,       "output_fs_hz":  OUTPUT_FS,
            "source_duration_sec": SOURCE_SAMPLES/SOURCE_FS,
            "output_duration_sec": OUTPUT_SAMPLES/OUTPUT_FS,
            "source_nan_count": src_nan, "source_inf_count": src_inf,
            "output_nan_count": out_nan, "output_inf_count": out_inf,
            "conversion_success": True, "error_code": None, "error_message": None,
            "source_rms":  ms["rms"],  "output_rms":  mo["rms"],  "rms_relerr":  rms_re,
            "source_energy_0_6khz": ms["energy_0_6khz"],
            "output_energy_0_6khz": mo["energy_0_6khz"],
            "energy_0_6khz_relerr": e_re,
            "source_dominant_freq": ms["dominant_freq"],
            "output_dominant_freq": mo["dominant_freq"],
            "dominant_freq_abserr": df_ae,
            "source_peak": ms["peak"], "output_peak": mo["peak"], "peak_relerr": peak_re,
            "source_crest": ms["crest"], "output_crest": mo["crest"], "crest_relerr": crest_re,
            "source_kurtosis": ms["kurtosis"], "output_kurtosis": mo["kurtosis"],
            "kurtosis_abserr": abs(mo["kurtosis"]-ms["kurtosis"]),
            "source_skew":     ms["skew"],      "output_skew":     mo["skew"],
            "skew_abserr":     abs(mo["skew"]-ms["skew"]),
            "rms_warning":     rms_re is not None and rms_re > RMS_WARN_THR,
            # 진단 대역 에너지
            **{f"src_{k}": ms[k] for k in ["e_0_2","e_2_4","e_4_6","e_6_8","e_8_10","e_10_nyq"]},
            **{f"out_{k}": mo[k] for k in ["e_0_2","e_2_4","e_4_6","e_6_8","e_8_10","e_10_nyq"]},
        }
        metric_rows.append(row)

        # .npy 저장
        arr_dir = ARRAYS_DIR / srow["split"] / srow["logical_bearing_id"]
        arr_dir.mkdir(parents=True, exist_ok=True)
        npy_p = arr_dir / f"{sid}.npy"
        np.save(npy_p, out.astype(np.float32))
        npy_manifest.append({
            "sample_id": sid, "npy_path": str(npy_p),
            "shape": str((2048,)), "dtype": "float32",
            "sha256_f64": sha256_bytes(out),
            "sha256_f32": sha256_bytes32(out),
        })

    except Exception as e:
        metric_rows.append({**base, "conversion_success": False,
                            "error_code": type(e).__name__,
                            "error_message": repr(e)[:400]})
        fail_rows.append({"sample_id": sid, "error": repr(e)})

    if (i+1) % 50 == 0:
        print(f"  [{i+1}/{TOTAL_CNT}]")

METRIC_DF = pd.DataFrame(metric_rows)
METRIC_DF.to_csv(OUTPUT_DIR / "apply_v1_1_confirmation_metrics.csv",
                 index=False, encoding="utf-8-sig")

# 대역 진단 저장
band_cols = ["sample_id","split","logical_bearing_id"] + \
            [f"src_{k}" for k in ["e_0_2","e_2_4","e_4_6","e_6_8","e_8_10","e_10_nyq"]] + \
            [f"out_{k}" for k in ["e_0_2","e_2_4","e_4_6","e_6_8","e_8_10","e_10_nyq"]] + \
            ["rms_relerr","energy_0_6khz_relerr","rms_warning"]
METRIC_DF[[c for c in band_cols if c in METRIC_DF.columns]].to_csv(
    OUTPUT_DIR / "apply_v1_1_band_energy_diagnostics.csv",
    index=False, encoding="utf-8-sig")

pd.DataFrame(npy_manifest).to_csv(
    OUTPUT_DIR / "converted_confirmation_manifest_v1_1.csv", index=False, encoding="utf-8-sig")

ok_df  = METRIC_DF[METRIC_DF["conversion_success"]==True].copy()

# split/record 집계
split_agg = ok_df.groupby("split").agg(
    n=("conversion_success","count"),
    rms_relerr_median=("rms_relerr",   safe_median),
    rms_relerr_p95=("rms_relerr",      safe_p95),
    energy_relerr_median=("energy_0_6khz_relerr", safe_median),
    energy_relerr_p95=("energy_0_6khz_relerr",    safe_p95),
    df_abserr_median=("dominant_freq_abserr",    safe_median),
    rms_warn=("rms_warning","sum"),
).reset_index()
split_agg.to_csv(OUTPUT_DIR / "apply_v1_1_summary_by_split.csv",
                 index=False, encoding="utf-8-sig")

rec_agg = ok_df.groupby("record_key").agg(
    n=("conversion_success","count"),
    rms_relerr_median=("rms_relerr",safe_median),
    energy_relerr_median=("energy_0_6khz_relerr",safe_median),
    rms_warn=("rms_warning","sum"),
).reset_index()
rec_agg.to_csv(OUTPUT_DIR / "apply_v1_1_summary_by_record.csv",
               index=False, encoding="utf-8-sig")

# RMS 이상치 Top50
rms_top50 = (
    ok_df[["sample_id","split","logical_bearing_id","csv_file_name",
           "rms_relerr","source_rms","output_rms","rms_warning"]]
    .sort_values("rms_relerr", ascending=False).head(50)
)
rms_top50.to_csv(OUTPUT_DIR / "apply_v1_1_rms_outlier_top50.csv",
                 index=False, encoding="utf-8-sig")

print(f"\n[EVAL] 성공={int(ok_df['conversion_success'].sum())}  실패={len(fail_rows)}")
display(split_agg[["split","n","rms_relerr_median","rms_relerr_p95","energy_relerr_median","rms_warn"]])

  [50/280]
  [100/280]
  [150/280]
  [200/280]
  [250/280]

[EVAL] 성공=270  실패=10


,split,n,rms_relerr_median,rms_relerr_p95,energy_relerr_median,rms_warn
0,FULL_TEST,100,0.039727,0.263452,0.001209,10
1,LEARNING,60,0.039432,0.557475,0.001274,24
2,TEST,110,0.043871,0.201734,0.001298,13


In [7]:
# ================================================================
# 셀 07 — 결정론 재현성 (LEARNING 4 + TEST 4 + FULL_TEST 4 = 12개)
# ================================================================

repro_rows = []
DETERMINISM_PASS = True

for sp in ["LEARNING", "TEST", "FULL_TEST"]:
    sp_df   = SAMPLE_DF[SAMPLE_DF["split"]==sp].reset_index(drop=True)
    indices = [int(round(i)) for i in np.linspace(0, len(sp_df)-1, REPRO_PER_SPLIT)]

    for idx in indices:
        srow  = sp_df.iloc[idx]
        fpath = Path(srow["absolute_path"])
        sid   = srow["sample_id"]
        result = {"sample_id": sid, "split": sp, "csv_file_name": srow["csv_file_name"],
                  "deterministic": False, "array_equal_f64": False,
                  "array_equal_f32": False, "max_abs_diff": None,
                  "sha256_f64_run1": None, "sha256_f64_run2": None,
                  "sha256_f32_run1": None, "sha256_f32_run2": None,
                  "error": None}
        try:
            src = load_h(fpath)
            r1  = convert_one(src)
            r2  = convert_one(src)
            eq64 = bool(np.array_equal(r1, r2))
            eq32 = bool(np.array_equal(r1.astype(np.float32), r2.astype(np.float32)))
            result.update({
                "deterministic":   eq64,
                "array_equal_f64": eq64,
                "array_equal_f32": eq32,
                "max_abs_diff":    float(np.max(np.abs(r1-r2))),
                "sha256_f64_run1": sha256_bytes(r1),
                "sha256_f64_run2": sha256_bytes(r2),
                "sha256_f32_run1": sha256_bytes32(r1),
                "sha256_f32_run2": sha256_bytes32(r2),
            })
            if not eq64: DETERMINISM_PASS = False
        except Exception as e:
            result["error"] = repr(e)
            DETERMINISM_PASS = False
        repro_rows.append(result)

REPRO_DF = pd.DataFrame(repro_rows)
REPRO_DF.to_csv(OUTPUT_DIR / "apply_v1_1_determinism_check.csv",
                index=False, encoding="utf-8-sig")
print(f"[REPRO] DETERMINISM_PASS = {DETERMINISM_PASS}  (테스트={len(REPRO_DF)}개)")
display(REPRO_DF[["sample_id","split","deterministic","max_abs_diff"]])

[REPRO] DETERMINISM_PASS = True  (테스트=12개)


,sample_id,split,deterministic,max_abs_diff
0,LEARNING_Bearing1_1_00,LEARNING,True,0.0
1,LEARNING_Bearing2_1_00,LEARNING,True,0.0
2,LEARNING_Bearing2_2_09,LEARNING,True,0.0
3,LEARNING_Bearing3_2_09,LEARNING,True,0.0
4,TEST_Bearing1_3_00,TEST,True,0.0
5,TEST_Bearing1_6_06,TEST,True,0.0
6,TEST_Bearing2_5_03,TEST,True,0.0
7,TEST_Bearing3_3_09,TEST,True,0.0
8,FULL_TEST_Bearing1_3_00,FULL_TEST,True,0.0
9,FULL_TEST_Bearing1_6_06,FULL_TEST,True,0.0


In [8]:
# ================================================================
# 셀 08 — 원본 불변성
# ================================================================

LOCK_TARGETS = [
    ("M0_REFERENCE",    "BASELINE_M0_frozen.json"),
    ("M0_REFERENCE",    "m0_baseline_result.csv"),
    ("BRIDGE_LOCKED",   str(PREP_DIR / "bridge_spec_locked.json")),
    ("BRIDGE_CONTRACT", str(APPLY_V1_DIR / "bridge_implementation_contract.json")),
    ("BRIDGE_PREP",     str(PREP_DIR / "FINAL_BRIDGE_PREP_SUMMARY.json")),
    ("APPLY_V1",        str(APPLY_V1_DIR / "APPLY_FINAL_SUMMARY.json")),
    ("FULL_MANIFEST",   str(SOURCE_MANIFEST_PATH)),
]

IMMUT_BEFORE = {}
for label, p_or_name in LOCK_TARGETS:
    p = Path(p_or_name) if Path(p_or_name).is_absolute() else None
    if p is None:
        hits = sorted(PROJECT_ROOT.rglob(p_or_name),
                      key=lambda x: x.stat().st_mtime, reverse=True)
        p = hits[0] if hits else None
    if p and p.exists():
        st = p.stat()
        IMMUT_BEFORE[str(p.resolve())] = {
            "label": label, "file_name": p.name,
            "size": int(st.st_size), "mtime_ns": int(st.st_mtime_ns),
            "sha256": sha256_file(p),
        }

# 280개 원본 CSV
for _, srow in SAMPLE_DF.iterrows():
    p = Path(srow["absolute_path"])
    if not p.exists(): continue
    st = p.stat()
    IMMUT_BEFORE[str(p.resolve())] = {
        "label": "SAMPLE_CSV", "file_name": p.name,
        "size":int(st.st_size), "mtime_ns":int(st.st_mtime_ns),
        "sha256": srow["source_sha256"],
    }

immut_rows = []
SOURCE_FILES_UNCHANGED = True

for abs_p, before in IMMUT_BEFORE.items():
    p = Path(abs_p)
    if not p.exists():
        SOURCE_FILES_UNCHANGED = False
        immut_rows.append({"absolute_path":abs_p,"label":before["label"],
                            "unchanged":False,"status":"MISSING_AFTER"})
        continue
    st = p.stat()
    s2, m2 = int(st.st_size), int(st.st_mtime_ns)
    if s2==before["size"] and m2==before["mtime_ns"]:
        sha2, ok = before["sha256"], True
    else:
        sha2 = sha256_file(p)
        ok   = (sha2 == before["sha256"])
    if not ok: SOURCE_FILES_UNCHANGED = False
    immut_rows.append({
        "absolute_path":abs_p, "label":before["label"], "file_name":before["file_name"],
        "size_before":before["size"],"size_after":s2,
        "sha256_before":before["sha256"],"sha256_after":sha2,
        "unchanged":ok, "status":"UNCHANGED" if ok else "MODIFIED",
    })

pd.DataFrame(immut_rows).to_csv(OUTPUT_DIR / "apply_v1_1_immutability_check.csv",
                                 index=False, encoding="utf-8-sig")
changed = sum(1 for r in immut_rows if not r["unchanged"])
print(f"[IMMUT] source_files_unchanged={SOURCE_FILES_UNCHANGED}  변경={changed}건")

[IMMUT] source_files_unchanged=True  변경=0건


In [10]:
conversion_success_count = int(METRIC_DF["conversion_success"].sum())
conversion_failure_count = int((~METRIC_DF["conversion_success"]).sum())

ok_df  = METRIC_DF[METRIC_DF["conversion_success"]==True].copy()

# 전체 품질 기준
q_vals = {
    "energy_0_6khz_relerr_median": safe_median(ok_df["energy_0_6khz_relerr"]),
    "energy_0_6khz_relerr_p95":    safe_p95(ok_df["energy_0_6khz_relerr"]),
    "dominant_freq_abserr_median": safe_median(ok_df["dominant_freq_abserr"]),
    "rms_relerr_median":           safe_median(ok_df["rms_relerr"]),
}
q_pass = {k:(v is not None and v<=QUALITY_CRITERIA[k]) for k,v in q_vals.items()}
QUALITY_PASS = all(q_pass.values())

cnt_by_split = SAMPLE_DF.groupby("split").size().to_dict()

# 최종 Gate (지시서 §11 required_final_checks)
required_final_checks = {
    "gate_pass":                 bool(GATE_PASS),
    "source_scope_pass":         bool(SOURCE_SCOPE_PASS),
    "source_csv_total_43369":    int(len(full_df)) == 43369,
    "vibration_total_39016":     int(len(vibration_df)) == 39016,
    "record_total_28":           int(SAMPLE_DF["record_key"].nunique()) == 28,
    "sample_total_280":          int(len(SAMPLE_DF)) == 280,
    "learning_sample_60":        int(cnt_by_split.get("LEARNING",0)) == 60,
    "test_sample_110":           int(cnt_by_split.get("TEST",0))     == 110,
    "full_test_sample_110":      int(cnt_by_split.get("FULL_TEST",0))== 110,
    "conversion_success_280":    conversion_success_count == 280,
    "conversion_failure_zero":   conversion_failure_count == 0,
    "quality_pass":              bool(QUALITY_PASS),
    "determinism_pass":          bool(DETERMINISM_PASS),
    "source_files_unchanged":    bool(SOURCE_FILES_UNCHANGED),
}

FINAL_PASS = all(required_final_checks.values())

def determine_failure_status(checks):
    if not checks.get("source_files_unchanged"): return "APPLY_V1_1_INVALID_SOURCE_MODIFIED"
    if not checks.get("determinism_pass"):       return "APPLY_V1_1_NONDETERMINISTIC"
    if not checks.get("gate_pass"):              return "APPLY_V1_1_BLOCKED_BY_GATE"
    if not checks.get("source_scope_pass"):      return "APPLY_V1_1_SOURCE_SCOPE_MISMATCH"
    if not (checks.get("sample_total_280") and
            checks.get("conversion_success_280") and
            checks.get("conversion_failure_zero")):
        return "APPLY_V1_1_SCHEMA_REVIEW_REQUIRED"
    if not checks.get("quality_pass"):           return "APPLY_V1_1_QUALITY_REVIEW_REQUIRED"
    return "APPLY_V1_1_REVIEW_REQUIRED"

if FINAL_PASS:
    apply_status = "APPLY_V1_1_PASS"
    next_action  = "READY_FOR_M0_BRIDGE_2048_FULL_CONVERT"
else:
    apply_status = determine_failure_status(required_final_checks)
    next_action  = "HOLD_FOR_REVIEW"

print("=" * 70)
print("최종 Gate:")
for k, v in required_final_checks.items():
    print(f"  {'✅' if v else '❌'} {k}: {v}")
print("\n품질 기준 (잠금 기준 그대로):")
for k, v in q_pass.items():
    val = q_vals[k]
    thr = QUALITY_CRITERIA[k]
    # Fix: Ensure .4f is only applied to numeric values
    print(f"  {'✅' if v else '❌'} {k}: {f'{val:.4f}' if val is not None else 'N/A'} <= {thr}")
print("=" * 70)
print(f"APPLY_STATUS : {apply_status}")
print(f"NEXT_ACTION  : {next_action}")
print("=" * 70)

# Fix: Ensure .4f is only applied to numeric values within _q_md construction
_q_md = "\n".join(
    f"| {k} | {f'{q_vals[k]:.4f}' if q_vals[k] is not None else 'N/A'} | {QUALITY_CRITERIA[k]} | {'✅' if q_pass[k] else '❌'} |"
    for k in QUALITY_CRITERIA
)

final_summary = {
    "version":                    VERSION,
    "created_at":                 datetime.now().isoformat(timespec="seconds"),
    "apply_status":               apply_status,
    "recommended_next_action":    next_action,
    "repair_reason":              "APPLY_SCOPE_MANIFEST_BUG",
    "previous_v1_sample_count":   60,
    "sample_count":               int(len(SAMPLE_DF)),
    "split_sample_counts":        {k:int(v) for k,v in cnt_by_split.items()},
    "record_count":               int(SAMPLE_DF["record_key"].nunique()),
    "source_csv_count":           int(len(full_df)),
    "vibration_csv_count":        int(len(vibration_df)),
    "conversion_success":         conversion_success_count,
    "conversion_failure":         conversion_failure_count,
    "quality_pass":               bool(QUALITY_PASS),
    "quality_values":             q_vals,
    "quality_criteria":           QUALITY_CRITERIA,
    "quality_pass_detail":        q_pass,
    "determinism_pass":           bool(DETERMINISM_PASS),
    "source_files_unchanged":     bool(SOURCE_FILES_UNCHANGED),
    "required_final_checks":      required_final_checks,
    "rms_warn_count":             int(ok_df["rms_warning"].sum()) if "rms_warning" in ok_df.columns else 0,
    "criteria_modified":          False,
    "bridge_parameters_modified": False,
    "test_full_test_used_for_selection": False,
    "full_dataset_conversion_performed": False,
    "output_directory":           str(OUTPUT_DIR),
}
write_json(OUTPUT_DIR / "APPLY_V1_1_FINAL_SUMMARY.json", final_summary)

summary_md = f"""# M0_BRIDGE_2048_APPLY_v1.1 Final Summary

## 최종 결과

| 항목 | 값 |
|:---|:---|
| **APPLY_STATUS** | `{apply_status}` |
| **NEXT ACTION** | `{next_action}` |
| 수정 원인 | `APPLY_SCOPE_MANIFEST_BUG` |
| v1 표본 | 60개 (LEARNING only) |
| v1.1 표본 | {len(SAMPLE_DF)}개 (28 records) |
| 변환 성공 | {conversion_success_count} |
| 변환 실패 | {conversion_failure_count} |
| 품질 PASS | `{QUALITY_PASS}` |
| 결정론 PASS | `{DETERMINISM_PASS}` |
| 원본 불변성 | `{SOURCE_FILES_UNCHANGED}` |

## 품질 기준 (잠금 기준 그대로)

| 지표 | 실제값 | 기준 | 판정 |
|:---|---:|---:|:---:|
{_q_md}

## 적용 내역

- `scipy.signal.resample_poly(x, up=4, down=5, window=("kaiser", 5.0), padtype="line")`
- TEST/FULL_TEST는 사후 적용 가능성 확인에만 사용 (방식·기준 변경 없음)
- RMS 기준은 median (p95 이상치는 경고 기록)
"""
with (OUTPUT_DIR / "APPLY_V1_1_FINAL_SUMMARY.md").open("w", encoding="utf-8") as f:
    f.write(summary_md)

# 필수 출력 확인
REQUIRED = [
    "apply_v1_1_gate_check.json", "full_source_scope_check.json",
    "vibration_source_manifest_39016.csv", "apply_v1_1_sample_manifest.csv",
    "apply_v1_1_confirmation_metrics.csv", "apply_v1_1_summary_by_record.csv",
    "apply_v1_1_summary_by_split.csv", "apply_v1_1_band_energy_diagnostics.csv",
    "apply_v1_1_rms_outlier_top50.csv", "apply_v1_1_determinism_check.csv",
    "apply_v1_1_immutability_check.csv", "converted_confirmation_manifest_v1_1.csv",
    "APPLY_V1_1_FINAL_SUMMARY.json", "APPLY_V1_1_FINAL_SUMMARY.md",
]
missing = [n for n in REQUIRED if not (OUTPUT_DIR/n).exists()]
if missing: print(f"⚠️  누락: {missing}")
else:        print("✅ 모든 필수 출력 파일 생성 완료")


최종 Gate:
  ✅ gate_pass: True
  ✅ source_scope_pass: True
  ✅ source_csv_total_43369: True
  ✅ vibration_total_39016: True
  ✅ record_total_28: True
  ✅ sample_total_280: True
  ✅ learning_sample_60: True
  ✅ test_sample_110: True
  ✅ full_test_sample_110: True
  ❌ conversion_success_280: False
  ❌ conversion_failure_zero: False
  ✅ quality_pass: True
  ✅ determinism_pass: True
  ✅ source_files_unchanged: True

품질 기준 (잠금 기준 그대로):
  ✅ energy_0_6khz_relerr_median: 0.0013 <= 0.05
  ✅ energy_0_6khz_relerr_p95: 0.0018 <= 0.1
  ✅ dominant_freq_abserr_median: 0.0000 <= 50.0
  ✅ rms_relerr_median: 0.0411 <= 0.05
APPLY_STATUS : APPLY_V1_1_SCHEMA_REVIEW_REQUIRED
NEXT_ACTION  : HOLD_FOR_REVIEW
✅ 모든 필수 출력 파일 생성 완료


In [12]:
# ================================================================
# 셀 10 — 결과 확인 (독립 실행 가능)
# ================================================================

import json, pandas as pd
from pathlib import Path

_pr = next(
    (c for c in [
        Path("/content/drive/MyDrive/Colab Notebooks/field_iis3dwb"),
        Path("/content/drive/My Drive/Colab Notebooks/field_iis3dwb"),
    ] if c.exists()), None
)
if _pr is None:
    print("PROJECT_ROOT 없음")
else:
    dirs = sorted(
        [p for p in (_pr/"bridge_outputs").glob("*_bridge_apply_v1_1")
         if p.is_dir() and (p/"APPLY_V1_1_FINAL_SUMMARY.json").exists()],
        key=lambda p: p.stat().st_mtime_ns, reverse=True,
    )
    if not dirs:
        print("APPLY_V1_1_FINAL_SUMMARY.json 없음 — 셀 02~09 먼저 실행")
    else:
        _d = dirs[0]
        with (_d/"APPLY_V1_1_FINAL_SUMMARY.json").open("r",encoding="utf-8") as f:
            s = json.load(f)

        print("=" * 70)
        print(f"APPLY_STATUS  : {s.get('apply_status')}")
        print(f"NEXT_ACTION   : {s.get('recommended_next_action')}")
        print(f"Samples       : {s.get('sample_count')} (v1=60 → v1.1={s.get('sample_count')})")
        print(f"Split counts  : {s.get('split_sample_counts')}")
        print(f"Conv success  : {s.get('conversion_success')}")
        print(f"Conv failure  : {s.get('conversion_failure')}")
        print(f"Quality PASS  : {s.get('quality_pass')}")
        print(f"Determinism   : {s.get('determinism_pass')}")
        print(f"Source immut. : {s.get('source_files_unchanged')}")
        print(f"RMS warn      : {s.get('rms_warn_count')}")
        print(f"Output dir    : {s.get('output_directory')}")
        print("=" * 70)

        print("\n품질 지표:")
        for k, v in s.get("quality_values",{}).items():
            thr = s.get("quality_criteria",{}).get(k,"?")
            ok  = s.get("quality_pass_detail",{}).get(k,False)
            print(f"  {'✅' if ok else '❌'} {k}: {f'{v:.4f}' if v is not None else 'N/A'} (≤{thr})")

        print("\nSplit별 집계:")
        _sp = _d / "apply_v1_1_summary_by_split.csv"
        if _sp.exists(): display(pd.read_csv(_sp))

        print("\nRMS 이상치 Top 5:")
        _rm = _d / "apply_v1_1_rms_outlier_top50.csv"
        if _rm.exists(): display(pd.read_csv(_rm).head())

APPLY_STATUS  : APPLY_V1_1_SCHEMA_REVIEW_REQUIRED
NEXT_ACTION   : HOLD_FOR_REVIEW
Samples       : 280 (v1=60 → v1.1=280)
Split counts  : {'FULL_TEST': 110, 'LEARNING': 60, 'TEST': 110}
Conv success  : 270
Conv failure  : 10
Quality PASS  : True
Determinism   : True
Source immut. : True
RMS warn      : 47
Output dir    : /content/drive/MyDrive/Colab Notebooks/field_iis3dwb/bridge_outputs/20260725_194318_bridge_apply_v1_1

품질 지표:
  ✅ energy_0_6khz_relerr_median: 0.0013 (≤0.05)
  ✅ energy_0_6khz_relerr_p95: 0.0018 (≤0.1)
  ✅ dominant_freq_abserr_median: 0.0000 (≤50.0)
  ✅ rms_relerr_median: 0.0411 (≤0.05)

Split별 집계:


,split,n,rms_relerr_median,rms_relerr_p95,energy_relerr_median,energy_relerr_p95,df_abserr_median,rms_warn
0,FULL_TEST,100,0.039727,0.263452,0.001209,0.001766,0.0,10
1,LEARNING,60,0.039432,0.557475,0.001274,0.001674,0.0,24
2,TEST,110,0.043871,0.201734,0.001298,0.001866,0.0,13



RMS 이상치 Top 5:


,sample_id,split,logical_bearing_id,csv_file_name,rms_relerr,source_rms,output_rms,rms_warning
0,LEARNING_Bearing2_2_07,LEARNING,Bearing2_2,acc_00620.csv,0.585472,0.642977,0.266532,True
1,LEARNING_Bearing2_2_06,LEARNING,Bearing2_2,acc_00532.csv,0.564856,0.735481,0.320040,True
2,LEARNING_Bearing2_1_03,LEARNING,Bearing2_1,acc_00304.csv,0.563704,0.732007,0.319372,True
3,LEARNING_Bearing2_2_05,LEARNING,Bearing2_2,acc_00443.csv,0.557147,0.819385,0.362867,True
4,LEARNING_Bearing2_2_08,LEARNING,Bearing2_2,acc_00709.csv,0.533375,0.627683,0.292892,True
